In [ ]:
# Standard Library
from pathlib import Path
import random

# Third-Party Libraries
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# PyTorch
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, WeightedRandomSampler

In [ ]:
# Define paths
ROOT = Path().resolve().parent

data_path = ROOT / 'data'

train_path = data_path / 'train'
test_path = data_path / 'test'
val_path = data_path / 'val'

print(f"Train path: {train_path}")
print(f"Test path:  {test_path}")
print(f"Val path:   {val_path}")

In [ ]:
# Image size ResNet expects
IMG_SIZE = 224

# Data augmentation + normalization for training
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Deterministic preprocessing for evaluation (no augmentation)
test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Class mapping based on folder names (ImageFolder auto-labels alphabetically)
#   NORMAL/ - label 0
#   PNEUMONIA/ - label 1

train_dataset = datasets.ImageFolder(
    root=train_path,
    transform=train_transforms
)

# Validation uses same preprocessing as test (no augmentation)
test_dataset = datasets.ImageFolder(
    root=test_path,
    transform=test_transforms
)

val_dataset = datasets.ImageFolder(
    root=val_path,
    transform=test_transforms
)

print(f"Classes: {train_dataset.classes}")
print(f"Class index mapping: {train_dataset.class_to_idx}")

print(f"\nTraining samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples:       {len(test_dataset):,}")

In [ ]:
# Compute raw class distribution from dataset labels
class_counts = np.bincount(train_dataset.targets)

# Display dataset imbalance (important for medical classification bias)
print(f"Class distribution:")
print(f"   NORMAL:    {class_counts[0]:,}")
print(f"   PNEUMONIA: {class_counts[1]:,}")
print(f"   Ratio:     {class_counts[1]/class_counts[0]:.1f}x more pneumonia")

# Inverse-frequency weighting to handle class imbalance
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_dataset.targets]
sample_weights = torch.tensor(sample_weights, dtype=torch.float32)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"Class weights:")
print(f"   NORMAL weight:    {class_weights[0]:.4f}")
print(f"   PNEUMONIA weight: {class_weights[1]:.4f}")
print(f"   Ratio:  {class_weights[0]/class_weights[1]:.1f}x more normal samples")

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset, 
    BATCH_SIZE, 
    sampler=sampler
)

# Evaluation loaders are deterministic (no shuffling)
test_loader = DataLoader(
    test_dataset,
    BATCH_SIZE,
    shuffle=False,
)

val_loader = DataLoader(
    val_dataset,
    BATCH_SIZE,
    shuffle=False,
)

In [ ]:
iterator = iter(train_loader)
images, labels = next(iterator)
print("Single batch inspection")
print(f"Image tensor shape: {images.shape}")
print(f"Batch size: {images.shape[0]}")
print(f"RGB channels: {images.shape[1]}")
print(f"Dimensions: {images.shape[2]}x{images.shape[3]}")
print(f"Normalized pixel range: {images.min()} - {images.max()}")

In [ ]:
normal_path = random.choice(list((train_path / 'NORMAL').glob('*.jpeg')))
pneumonia_path = random.choice(list((train_path / 'PNEUMONIA').glob('*.jpeg')))

normal_image = Image.open(normal_path).convert('RGB')
pneumonia_image = Image.open(pneumonia_path).convert('RGB')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Chest X-Ray Samples', fontsize=16, fontweight='bold')

for i in range (3):
    transformed_N = train_transforms(normal_image)
    transformed_P = train_transforms(pneumonia_image)

    axes[0, i].imshow(transformed_N.permute(1, 2, 0).numpy())
    axes[0, i].set_title(f'NORMAL\n{normal_path.stem[:30]}', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(transformed_P.permute(1, 2, 0).numpy())
    axes[1, i].set_title(f'PNEUMONIA\n{pneumonia_path.stem[:30]}', fontsize=10)
    axes[1, i].axis('off')

